# 03 — Fixed-256-D layer-set selection and representation ablations

This notebook consumes the completed frozen SfM cache. It never re-extracts
DINO features. Its output dimension is fixed at 256 so that layer-set selection
is not confounded by descriptor capacity.

It compares frozen Final CLS, a trainable final-CLS projection, multi-level CLS
concatenation, and global-local fusion with Uniform, Static, and Dynamic layer
weighting. The student manually records the selected layer set after reviewing
SfM-only results. This notebook does not choose a final model and must not use
RevisitOP.


In [ ]:
# Run this first in every fresh Colab runtime. It intentionally does not use PYTHONPATH.
from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys

REPO_URL = os.environ.get('CBIR_REPO_URL', 'https://github.com/armin-faraji/LightweightCBIR.git')
REPO_REVISION = os.environ.get('CBIR_REPO_REVISION', 'main')
PROJECT_ROOT = Path('/content/lightweight-cbir')
if not PROJECT_ROOT.is_dir():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', REPO_REVISION], check=True)
os.chdir(PROJECT_ROOT)
required_project_files = (
    PROJECT_ROOT / 'pyproject.toml',
    PROJECT_ROOT / 'src' / 'cbir' / '__init__.py',
    PROJECT_ROOT / 'src' / 'cbir' / 'artifacts.py',
)
missing_project_files = [
    str(path.relative_to(PROJECT_ROOT)) for path in required_project_files if not path.is_file()
]
if missing_project_files:
    raise RuntimeError(
        'The cloned repository does not contain the required project code: '
        + ', '.join(missing_project_files)
    )
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', '--force-reinstall', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'h5py', 'scipy', 'PyYAML', 'tqdm', 'matplotlib', 'Pillow'], check=True)

importlib.invalidate_caches()
import cbir
print('cbir package:', cbir.__file__)

from cbir.artifacts import create_artifact_run, make_artifact_run_id
from cbir.cloud import mount_colab_drive, runtime_report, write_runtime_report
from cbir.config import config_to_dict, load_project_config
from cbir.utils import stable_hash

PERSISTENT_ROOT = mount_colab_drive() / 'lightweight-cbir'
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('TORCH_HOME', str(PERSISTENT_ROOT / 'torch_hub'))
CONFIG_PATH = Path('configs/colab.yaml')
cfg = load_project_config(CONFIG_PATH)
FIXED_OUTPUT_DIM = 256
cfg = replace(cfg, fusion=replace(cfg.fusion, output_dim=FIXED_OUTPUT_DIM))
environment = runtime_report(project_root=PROJECT_ROOT)
config_fingerprint = stable_hash(config_to_dict(cfg))
RUN_ID = make_artifact_run_id(
    notebook='03', git_sha=environment['git_sha'], config_fingerprint=config_fingerprint
)
ARTIFACT_RUN = create_artifact_run(
    PROJECT_ROOT / 'outputs',
    '03',
    run_id=RUN_ID,
    metadata={'git_sha': environment['git_sha'], 'config_fingerprint': config_fingerprint},
)
LOCAL_OUTPUT_DIR = ARTIFACT_RUN.local_dir
DRIVE_OUTPUT_ROOT = PERSISTENT_ROOT / 'notebook_outputs'
write_runtime_report(
    LOCAL_OUTPUT_DIR,
    project_root=PROJECT_ROOT,
    extra={'notebook': '03', 'config': str(CONFIG_PATH), 'fixed_output_dim': FIXED_OUTPUT_DIM},
)
shutil.copy2(CONFIG_PATH, ARTIFACT_RUN.path_for('config.yaml'))
ARTIFACT_RUN.write_json('effective_config.json', config_to_dict(cfg))
print('Project:', PROJECT_ROOT)
print('Artifacts:', LOCAL_OUTPUT_DIR)
print('Layer-selection descriptor dimension:', FIXED_OUTPUT_DIM)


## Restore metadata and the completed frozen feature cache


In [ ]:
from cbir.cache import FeatureShardReader
from cbir.cloud import publish_file, stage_file
from cbir.data.sfm import Sfm30kMetadata
from cbir.workflow import restore_complete_sfm_cache

LOCAL_SFM_ROOT = cfg.sfm.metadata_path.parent
DRIVE_SFM_ROOT = PERSISTENT_ROOT / 'datasets' / 'sfm30k'
metadata_files = (cfg.sfm.metadata_path, cfg.sfm.names_clusters_path)
LOCAL_SFM_ROOT.mkdir(parents=True, exist_ok=True)
if not all(path is not None and path.is_file() for path in metadata_files):
    if all(path is not None and (DRIVE_SFM_ROOT / path.name).is_file() for path in metadata_files):
        for path in metadata_files:
            assert path is not None
            stage_file(DRIVE_SFM_ROOT / path.name, path)
    else:
        subprocess.run(
            [sys.executable, 'scripts/prepare_sfm30k.py', '--config', str(CONFIG_PATH), '--image-source', 'none'],
            cwd=PROJECT_ROOT,
            check=True,
        )
        for path in metadata_files:
            assert path is not None
            publish_file(path, DRIVE_SFM_ROOT / path.name)
if cfg.sfm.names_clusters_path is None:
    raise ValueError('configs/colab.yaml must define sfm.names_clusters_path')
metadata = Sfm30kMetadata.from_official_files(cfg.sfm.metadata_path, cfg.sfm.names_clusters_path)
cache_location = restore_complete_sfm_cache(cfg, metadata)
reader = FeatureShardReader(cache_location.local_dir)
if set(reader.image_ids) != set(metadata.image_ids()):
    raise RuntimeError('restored cache does not contain exactly the full SfM-30k protocol image IDs')
val_ids = metadata.image_ids('val')
val_cases = metadata.build_validation_cases()
ARTIFACT_RUN.write_json('cache_restore.json', {
    'cache_name': cache_location.cache_name,
    'fingerprint': cache_location.fingerprint,
    'local_dir': str(cache_location.local_dir),
    'drive_dir': None if cache_location.drive_dir is None else str(cache_location.drive_dir),
})
print('Using restored cache:', cache_location.local_dir)


## B0 — Frozen Final CLS baseline


In [ ]:
from cbir.evaluation import evaluate_sfm_verified_pairs, final_cls_descriptors_from_cache

baseline = final_cls_descriptors_from_cache(reader, val_ids)
baseline_report = evaluate_sfm_verified_pairs(baseline, val_ids, val_cases)
baseline_metrics = {
    'label': 'Final CLS — 384-D (frozen)',
    'descriptor_dimension': 384,
    'recall_at_1': baseline_report.recall_at_1,
    'recall_at_5': baseline_report.recall_at_5,
    'recall_at_10': baseline_report.recall_at_10,
    'mrr': baseline_report.mrr,
}
ARTIFACT_RUN.write_json('baseline_metrics.json', baseline_metrics)
print(baseline_metrics)


## Declare fixed-256-D experiments

The four candidate layer sets are tested with multi-level CLS concatenation and
all three global-local layer-weighting variants. The final CLS projection is a
single 256-D trained control.


In [ ]:
from dataclasses import replace

from cbir.cloud import publish_file
from cbir.config import config_to_dict, train_fingerprint
from cbir.fusion import build_descriptor_head
from cbir.plotting import HorizontalReference, SeriesData, plot_series
from cbir.training import HeadTrainer
from cbir.utils import atomic_write_json, seed_everything

# This is the only experiment declaration for Notebook 03. All runs use 256-D
# descriptors. Layer numbers remain one-based in the notebook/report and are
# converted only when building the DINO configuration.
LAYER_SETS_ONE_BASED = (
    (4, 8, 12),
    (10, 11, 12),
    (8, 10, 12),
    (6, 9, 12),
)

EXPERIMENT_SPECS = (
    {
        'name': 'final_cls_projection_256',
        'label': 'Final CLS + projection — 256-D',
        'head_kind': 'final_cls_projection',
        'gate_mode': None,
        'layers_one_based': (12,),
    },
    *tuple(
        spec
        for layers in LAYER_SETS_ONE_BASED
        for spec in (
            {
                'name': 'cls_concat_' + '_'.join(map(str, layers)),
                'label': f'Multi-level CLS concatenation ({", ".join(map(str, layers))}) — 256-D',
                'head_kind': 'cls_concat',
                'gate_mode': None,
                'layers_one_based': layers,
            },
            {
                'name': 'uniform_layer_weighting_' + '_'.join(map(str, layers)),
                'label': f'Uniform layer weighting ({", ".join(map(str, layers))}) — 256-D',
                'head_kind': 'global_local',
                'gate_mode': 'uniform',
                'layers_one_based': layers,
            },
            {
                'name': 'static_layer_weighting_' + '_'.join(map(str, layers)),
                'label': f'Static layer weighting ({", ".join(map(str, layers))}) — 256-D',
                'head_kind': 'global_local',
                'gate_mode': 'static',
                'layers_one_based': layers,
            },
            {
                'name': 'dynamic_layer_weighting_' + '_'.join(map(str, layers)),
                'label': f'Dynamic layer weighting ({", ".join(map(str, layers))}) — 256-D',
                'head_kind': 'global_local',
                'gate_mode': 'dynamic',
                'layers_one_based': layers,
            },
        )
    ),
)
EXPERIMENTS_BY_NAME = {spec['name']: spec for spec in EXPERIMENT_SPECS}
EXPERIMENT_NAMES = tuple(spec['name'] for spec in EXPERIMENT_SPECS)
if len(EXPERIMENTS_BY_NAME) != len(EXPERIMENT_SPECS):
    raise ValueError('Experiment names must be unique')
for spec in EXPERIMENT_SPECS:
    layers_one_based = tuple(int(layer) for layer in spec['layers_one_based'])
    if len(set(layers_one_based)) != len(layers_one_based):
        raise ValueError(f"{spec['name']} has duplicate layer indices")
    if min(layers_one_based) < 1 or max(layers_one_based) > cfg.backbone.num_blocks:
        raise ValueError(f"{spec['name']} has layers outside the backbone range")
    if spec['head_kind'] == 'global_local' and spec['gate_mode'] not in {'uniform', 'static', 'dynamic'}:
        raise ValueError(f"{spec['name']} has invalid layer weighting")
    if spec['head_kind'] != 'global_local' and spec['gate_mode'] is not None:
        raise ValueError(f"{spec['name']} must not define layer weighting")

PERSISTENT_CHECKPOINT_ROOT = PERSISTENT_ROOT / 'checkpoints' / '03' / ARTIFACT_RUN.run_id
histories = {}
ablation_results = {}


def zero_based_layers(spec):
    return tuple(int(layer) - 1 for layer in spec['layers_one_based'])


def write_experiment_summary():
    ARTIFACT_RUN.write_json('experiment_summary.json', {
        'baseline': baseline_metrics,
        'fixed_output_dim': FIXED_OUTPUT_DIM,
        'experiment_specs': list(EXPERIMENT_SPECS),
        'experiments': ablation_results,
    })


def record_experiment(experiment_name):
    if experiment_name in ablation_results:
        print('Already completed in this runtime:', experiment_name)
        return histories[experiment_name], ablation_results[experiment_name]
    spec = EXPERIMENTS_BY_NAME[experiment_name]
    fusion_cfg = replace(
        cfg.fusion,
        layer_indices=zero_based_layers(spec),
        output_dim=FIXED_OUTPUT_DIM,
        head_kind=spec['head_kind'],
        gate_mode=spec['gate_mode'],
    )
    run_fingerprint = train_fingerprint(
        cache_fingerprint=reader.manifest.fingerprint,
        fusion=fusion_cfg,
        training=cfg.training,
    )
    local_checkpoint_dir = ARTIFACT_RUN.path_for(Path('checkpoints') / experiment_name)
    persistent_checkpoint = PERSISTENT_CHECKPOINT_ROOT / experiment_name / 'best.pt'

    def publish_best_checkpoint(local_path):
        publish_file(local_path, persistent_checkpoint)

    seed_everything(cfg.training.seed)
    head = build_descriptor_head(fusion_cfg)
    trainable_parameter_count = sum(parameter.numel() for parameter in head.parameters())
    trainer = HeadTrainer(
        head=head,
        reader=reader,
        train_pairs=metadata.train_pairs,
        fusion_config=fusion_cfg,
        training_config=cfg.training,
        validation_cases=val_cases,
        validation_image_ids=val_ids,
        output_dir=local_checkpoint_dir,
        checkpoint_callback=publish_best_checkpoint,
    )
    history = trainer.fit()
    if history.best_checkpoint is None or history.best_epoch is None:
        raise RuntimeError(f'{experiment_name} did not produce a selected checkpoint')
    best_epoch_metrics = history.epochs[history.best_epoch]
    result = {
        'experiment_label': spec['label'],
        'head_kind': spec['head_kind'],
        'layer_weighting': spec['gate_mode'],
        'layer_indices_one_based': list(spec['layers_one_based']),
        'fusion_config': config_to_dict(fusion_cfg),
        'training_config': config_to_dict(cfg.training),
        'run_fingerprint': run_fingerprint,
        'trainable_parameter_count': trainable_parameter_count,
        'best_epoch': history.best_epoch + 1,
        'best_metric': history.best_metric,
        'best_epoch_metrics': best_epoch_metrics,
        'local_checkpoint': str(history.best_checkpoint),
        'persistent_checkpoint': str(persistent_checkpoint),
    }
    histories[experiment_name] = history
    ablation_results[experiment_name] = result
    write_experiment_summary()
    print(f"{spec['label']}: best SfM validation R@1={history.best_metric:.4f} at epoch {history.best_epoch + 1}")
    return history, result


def run_group(layers_one_based):
    suffix = '_'.join(map(str, layers_one_based))
    names = (
        'cls_concat_' + suffix,
        'uniform_layer_weighting_' + suffix,
        'static_layer_weighting_' + suffix,
        'dynamic_layer_weighting_' + suffix,
    )
    return [record_experiment(name) for name in names]

write_experiment_summary()
print('Declared trainable experiments:', len(EXPERIMENT_NAMES))


## B1 — Final CLS + projection — 256-D


In [ ]:
record_experiment('final_cls_projection_256')


## Candidate layer set: (4, 8, 12)


In [ ]:
run_group((4, 8, 12))


## Candidate layer set: (10, 11, 12)


In [ ]:
run_group((10, 11, 12))


## Candidate layer set: (8, 10, 12)


In [ ]:
run_group((8, 10, 12))


## Candidate layer set: (6, 9, 12)


In [ ]:
run_group((6, 9, 12))


## Audit, plots, and dynamic-layer-weighting diagnostics


In [ ]:
from itertools import combinations

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as functional
from IPython.display import display

from cbir.cache import sha256_file
from cbir.fusion import MultiLevelGlobalLocalFusion

missing_experiments = [name for name in EXPERIMENT_NAMES if name not in ablation_results]
if missing_experiments:
    raise RuntimeError('Run every declared experiment before audit: ' + ', '.join(missing_experiments))

# The frozen B0 line belongs on the key layer-selection R@1 plot. It has no
# training epochs, so it is a horizontal labelled reference rather than a curve.
primary_uniform_names = [
    'uniform_layer_weighting_' + '_'.join(map(str, layers))
    for layers in LAYER_SETS_ONE_BASED
]
primary_uniform_series = {
    EXPERIMENTS_BY_NAME[name]['label']: SeriesData(
        x=[int(epoch['epoch']) + 1 for epoch in histories[name].epochs],
        y=[epoch['val_recall_at_1'] for epoch in histories[name].epochs],
    )
    for name in primary_uniform_names
}
figure_selection, _ = plot_series(
    primary_uniform_series,
    title='SfM validation R@1 for fixed-256-D layer-set selection',
    xlabel='Epoch',
    ylabel='R@1',
    legend_title='Descriptor configuration',
    horizontal_references={
        baseline_metrics['label']: HorizontalReference(baseline_metrics['recall_at_1'])
    },
    fig_size=(12, 6),
    save_path=ARTIFACT_RUN.path_for('figures/layer_selection_validation_r1.png'),
)
display(figure_selection)
plt.close(figure_selection)

# Compare the new CLS-only controls directly with the primary uniform
# global-local representation for every candidate layer set.
representation_names = [
    'final_cls_projection_256',
    *['cls_concat_' + '_'.join(map(str, layers)) for layers in LAYER_SETS_ONE_BASED],
    *primary_uniform_names,
]
representation_series = {
    EXPERIMENTS_BY_NAME[name]['label']: SeriesData(
        x=[int(epoch['epoch']) + 1 for epoch in histories[name].epochs],
        y=[epoch['val_recall_at_1'] for epoch in histories[name].epochs],
    )
    for name in representation_names
}
figure_representation, _ = plot_series(
    representation_series,
    title='CLS-only versus global-local representation ablations at 256-D',
    xlabel='Epoch',
    ylabel='R@1',
    legend_title='Descriptor configuration',
    horizontal_references={
        baseline_metrics['label']: HorizontalReference(baseline_metrics['recall_at_1'])
    },
    fig_size=(14, 7),
    save_path=ARTIFACT_RUN.path_for('figures/representation_ablation_validation_r1.png'),
)
display(figure_representation)
plt.close(figure_representation)

all_global_local_names = [
    name for name, spec in EXPERIMENTS_BY_NAME.items()
    if spec['head_kind'] == 'global_local'
]
all_global_local_series = {
    EXPERIMENTS_BY_NAME[name]['label']: SeriesData(
        x=[int(epoch['epoch']) + 1 for epoch in histories[name].epochs],
        y=[epoch['val_recall_at_1'] for epoch in histories[name].epochs],
    )
    for name in all_global_local_names
}
figure_weighting, _ = plot_series(
    all_global_local_series,
    title='Global-local layer-weighting ablations at 256-D',
    xlabel='Epoch',
    ylabel='R@1',
    legend_title='Descriptor configuration',
    horizontal_references={
        baseline_metrics['label']: HorizontalReference(baseline_metrics['recall_at_1'])
    },
    fig_size=(14, 7),
    save_path=ARTIFACT_RUN.path_for('figures/global_local_weighting_validation_r1.png'),
)
display(figure_weighting)
plt.close(figure_weighting)

summary_rows = [
    {
        'name': 'final_cls_384_frozen',
        **baseline_metrics,
        'head_kind': 'frozen_final_cls',
        'layer_weighting': None,
        'layer_indices_one_based': [12],
        'trainable_parameter_count': 0,
    },
    *[
        {'name': name, **result}
        for name, result in ablation_results.items()
    ],
]
ARTIFACT_RUN.write_json('result_table.json', summary_rows)
print({row['name']: row.get('best_metric', row.get('recall_at_1')) for row in summary_rows})

# LEMG diagnostics are intentionally restricted to dynamic global-local heads.
dynamic_diagnostics = {}
sample_image_ids = tuple(val_ids[: min(512, len(val_ids))])
for name in all_global_local_names:
    spec = EXPERIMENTS_BY_NAME[name]
    if spec['gate_mode'] != 'dynamic':
        continue
    result = ablation_results[name]
    checkpoint = torch.load(result['persistent_checkpoint'], map_location='cpu', weights_only=False)
    fusion_payload = dict(checkpoint['fusion_config'])
    fusion_payload['layer_indices'] = tuple(fusion_payload['layer_indices'])
    from cbir.config import FusionConfig
    fusion_cfg = FusionConfig(**fusion_payload)
    head = MultiLevelGlobalLocalFusion.from_config(fusion_cfg).to(cfg.training.device).eval()
    head.load_state_dict(checkpoint['model_state_dict'])
    sample = reader.fetch(sample_image_ids, layer_indices=fusion_cfg.layer_indices, local_kind=fusion_cfg.local_kind)
    sample = {key: value.to(cfg.training.device) for key, value in sample.items()}
    with torch.inference_mode():
        _, diagnostics = head(sample['cls'], sample['local'], sample['entropy'], return_diagnostics=True)
    assert diagnostics.layer_weights is not None
    assert diagnostics.entropy_penalty_scale is not None
    dynamic_diagnostics[name] = {
        'label': spec['label'],
        'entropy_penalty_scale': float(diagnostics.entropy_penalty_scale.detach().cpu()),
        'weight_mean': [float(value) for value in diagnostics.layer_weights.mean(dim=0).detach().cpu()],
        'weight_std': [float(value) for value in diagnostics.layer_weights.std(dim=0, unbiased=False).detach().cpu()],
    }
ARTIFACT_RUN.write_json('dynamic_layer_weighting_diagnostics.json', dynamic_diagnostics)
write_experiment_summary()


## Manually record the SfM-selected layer set

Set the variable in the following cell only after inspecting the SfM-only
results. This is intentionally a manual research decision; it does not apply an
automatic tie-breaking rule. It writes an intermediate layer-selection record,
not a final-model lock.


In [ ]:
from cbir.utils import atomic_write_json

# Select the layer set manually after inspecting only the SfM results above.
# Example: SELECTED_LAYER_SET_ONE_BASED = (8, 10, 12)
SELECTED_LAYER_SET_ONE_BASED = None

if SELECTED_LAYER_SET_ONE_BASED is None:
    print('No layer set recorded yet. Inspect the SfM-only outputs, set SELECTED_LAYER_SET_ONE_BASED, then rerun this cell.')
else:
    chosen_layers = tuple(int(layer) for layer in SELECTED_LAYER_SET_ONE_BASED)
    if chosen_layers not in LAYER_SETS_ONE_BASED:
        raise ValueError('Choose one of the declared layer sets: ' + repr(LAYER_SETS_ONE_BASED))
    layer_selection = {
        'selected_on': 'Manual student decision after SfM-30k validation inspection only',
        'notebook_run_id': ARTIFACT_RUN.run_id,
        'cache_fingerprint': reader.manifest.fingerprint,
        'selected_layer_indices_one_based': list(chosen_layers),
        'available_layer_sets_one_based': [list(layers) for layers in LAYER_SETS_ONE_BASED],
        'fixed_output_dim': FIXED_OUTPUT_DIM,
        'baseline': baseline_metrics,
        'experiment_results': ablation_results,
    }
    layer_selection_path = PERSISTENT_ROOT / 'selected' / 'layer_selection.json'
    atomic_write_json(layer_selection_path, layer_selection)
    ARTIFACT_RUN.write_json('layer_selection.json', layer_selection)
    print('Recorded manual SfM-only layer selection:', chosen_layers)


## Publish Notebook 03 outputs to Drive


In [ ]:
published_output = ARTIFACT_RUN.publish(DRIVE_OUTPUT_ROOT)
print('Validated notebook artifacts published to:', published_output)
